# Bài 57: Depth Anything

Bài 56 đã cho thấy điểm yếu trung tâm của độ sâu đơn mắt: một mạng được huấn luyện trên một tập dữ liệu nhỏ, hẹp chỉ học được các tiên nghiệm đúng *bên trong* phân phối của tập dữ liệu đó, và bị đánh lừa một cách tự tin bên ngoài nó. **Depth Anything** (<a href="../references.html#yang-2024-depthanything">Yang et al., 2024</a>) là một mô hình nền tảng độ sâu đơn mắt được xây dựng đặc biệt để thu hẹp khoảng cách đó, dùng **tự huấn luyện trên dữ liệu không nhãn (self-training on unlabeled data)**: một mô hình giáo viên gán nhãn giả (pseudo-label) cho một kho ảnh không nhãn khổng lồ, và một học sinh huấn luyện trên dữ liệu có nhãn cộng với các nhãn giả đó, ở một quy mô (hơn 60 triệu ảnh không nhãn) mà không tập dữ liệu độ sâu chú thích thủ công nào tiến gần được. Bài này xây dựng quy trình đó &mdash; và kết quả của nó là một phát hiện tiêu cực thực sự, hữu ích: gán nhãn giả từ một giáo viên bị thiên lệch, tự nó, không khắc phục được sự thiên lệch. Hiểu *tại sao* là điều thực sự giải thích những gì Depth Anything làm đúng mà một phiên bản ngây thơ của cùng ý tưởng không làm được.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Thiết lập: một tập có nhãn nhỏ, một kho không nhãn lớn, một khoảng cách phân phối

Dùng lại các cảnh độ sâu theo tín hiệu kích thước ở Bài 56. Tập **có nhãn** nhỏ (20 cảnh) và chỉ bao phủ một khoảng độ sâu hẹp (0.4-0.6). Kho **không nhãn** lớn hơn nhiều (150 cảnh, không có nhãn độ sâu — chỉ có ảnh) và, giống như các bộ sưu tập ảnh không nhãn trong thế giới thực, trải rộng *toàn bộ* khoảng điều kiện (độ sâu 0.2-0.9). Tập **kiểm tra** cũng khớp với toàn bộ khoảng đó — cố tình bao gồm các độ sâu mà tập có nhãn chưa bao giờ cho thấy.

In [ ]:
SIZE = 32

def make_scene(rng, size=SIZE, n_objects=4, depth_range=(0.2, 0.9)):
    img = np.zeros((size, size), dtype=np.float32)
    depth = np.full((size, size), 1.0, dtype=np.float32)
    depths_used = rng.uniform(*depth_range, n_objects)
    depths_used.sort()
    for d in depths_used[::-1]:
        r = max(2, int(6 * (1 - d) + 1))
        cy = int(size * (0.3 + 0.6 * d))
        cx = rng.integers(r, size - r)
        yy, xx = np.mgrid[0:size, 0:size]
        m = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
        img[m] = 0.8
        depth[m] = d
    img = np.clip(img + rng.normal(0, 0.03, img.shape), 0, 1).astype(np.float32)
    return img, depth

label_rng = np.random.default_rng(10)
N_LABELED = 20
X_labeled, D_labeled = [], []
for _ in range(N_LABELED):
    im, d = make_scene(label_rng, n_objects=label_rng.integers(2, 4), depth_range=(0.4, 0.6))
    X_labeled.append(im); D_labeled.append(d)
X_labeled, D_labeled = np.array(X_labeled, dtype=np.float32), np.array(D_labeled, dtype=np.float32)

unlabeled_rng = np.random.default_rng(20)
N_UNLABELED = 150
X_unlabeled = []
for _ in range(N_UNLABELED):
    im, _ = make_scene(unlabeled_rng, n_objects=unlabeled_rng.integers(2, 6), depth_range=(0.2, 0.9))
    X_unlabeled.append(im)
X_unlabeled = np.array(X_unlabeled, dtype=np.float32)

test_rng = np.random.default_rng(30)
N_TEST = 100
X_test, D_test = [], []
for _ in range(N_TEST):
    im, d = make_scene(test_rng, n_objects=test_rng.integers(2, 6), depth_range=(0.2, 0.9))
    X_test.append(im); D_test.append(d)
X_test, D_test = np.array(X_test, dtype=np.float32), np.array(D_test, dtype=np.float32)

print(f'labeled:   {N_LABELED} scenes, depth range [0.4, 0.6]')
print(f'unlabeled: {N_UNLABELED} scenes, depth range [0.2, 0.9], no depth labels')
print(f'test:      {N_TEST} scenes, depth range [0.2, 0.9]')

## Giáo viên, nhãn giả, học sinh

Huấn luyện một **giáo viên** (U-Net ở Bài 42) chỉ trên tập có nhãn nhỏ. Dùng nó để gán nhãn giả cho mọi ảnh trong kho không nhãn. Huấn luyện một **học sinh**, cùng kiến trúc, trên tập có nhãn cộng với kho đã gán nhãn giả kết hợp lại — chính xác là công thức cốt lõi của Depth Anything.

In [ ]:
class UNetTiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec1 = nn.Sequential(nn.Conv2d(32 + 16, 16, 3, padding=1), nn.ReLU())
        self.out = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        f1 = self.enc1(x)
        f2 = self.enc2(self.pool(f1))
        d1 = self.dec1(torch.cat([self.up(f2), f1], dim=1))
        return torch.sigmoid(self.out(d1)).squeeze(1)

def train_model(X, D, seed, epochs=200, lr=0.01):
    torch.manual_seed(seed)
    model = UNetTiny()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt, Dt = torch.tensor(X).unsqueeze(1), torch.tensor(D)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.mse_loss(model(Xt), Dt)
        loss.backward()
        opt.step()
    return model

def eval_mae(model, X, D):
    with torch.no_grad():
        pred = model(torch.tensor(X).unsqueeze(1))
    return (pred - torch.tensor(D)).abs().mean().item()

teacher = train_model(X_labeled, D_labeled, seed=0)
teacher_mae = eval_mae(teacher, X_test, D_test)
print(f'teacher (small labeled set only) test MAE: {teacher_mae:.4f}')

with torch.no_grad():
    pseudo_labels = teacher(torch.tensor(X_unlabeled).unsqueeze(1)).numpy()

X_combined = np.concatenate([X_labeled, X_unlabeled])
D_combined = np.concatenate([D_labeled, pseudo_labels])
student = train_model(X_combined, D_combined, seed=1)
student_mae = eval_mae(student, X_test, D_test)
print(f'student (labeled + pseudo-labeled) test MAE: {student_mae:.4f}')

## Tại sao tự huấn luyện không giúp ích?

Sai số của học sinh về cơ bản không thay đổi so với của giáo viên. Đây không phải lỗi — đó là kết quả được kỳ vọng của tự huấn luyện ngây thơ, và đáng để hiểu chính xác tại sao. Các nhãn giả cho kho không nhãn đến *hoàn toàn* từ giáo viên, và giáo viên chưa bao giờ được cho thấy một độ sâu nào ngoài [0.4, 0.6]. Với bất kỳ cảnh không nhãn nào có độ sâu thực, chẳng hạn, 0.8, nhãn giả của giáo viên không phải là một phiên bản nhiễu của đáp án đúng — nó là chính lỗi ngoại suy có hệ thống của giáo viên từ thí nghiệm đánh lừa ở Bài 56, được áp dụng một cách tự tin rồi trao cho học sinh như thể đó là dữ liệu chuẩn. Huấn luyện trên đó chỉ dạy học sinh tái tạo lỗi của giáo viên một cách hiệu quả hơn. Không có thông tin thực sự mới nào về các độ sâu trong [0.6, 0.9] từng đi vào quy trình — tự huấn luyện phân phối lại những gì một mô hình đã biết, và không thể tạo ra những gì nó không biết.

## Điều gì thực sự khiến Depth Anything hoạt động

Khoảng cách giữa thất bại đồ chơi này và thành công thực sự, đã được ghi nhận rõ của Depth Anything đến từ ba thành phần mà thiết lập của bài này cố tình thiếu:

- **Độ bao phủ phân phối thực sự.** Kho không nhãn của Depth Anything không phải là một phân phối tổng hợp hẹp chỉ lệch nhẹ khỏi tập có nhãn — nó là 62 triệu ảnh *thật* trải rộng trên một phạm vi khổng lồ các cảnh, điều kiện ánh sáng và loại đối tượng. Lỗi của giáo viên trên bất kỳ ảnh không nhãn đơn lẻ nào vẫn là lỗi thật, nhưng khi lấy trung bình trên lượng đa dạng thực sự đó, tín hiệu huấn luyện của học sinh ít bị thiên lệch có hệ thống hơn nhiều so với kho đồ chơi 150 ảnh, một tín hiệu duy nhất của bài này.
- **Một mạng xương sống tiền huấn luyện mạnh.** Giáo viên không phải là một CNN nhỏ huấn luyện từ đầu trên một nhúm mẫu có nhãn — nó bắt đầu từ một mạng xương sống DINOv2 (Bài 50) đã mang các tiên nghiệm thị giác phong phú, đa năng học được từ dữ liệu không nhãn ở một quy mô hoàn toàn khác, trước khi từng thấy một nhãn độ sâu nào.
- **Nhiễu loạn mạnh trong khi huấn luyện học sinh.** Depth Anything đặc biệt đưa vào các phép tăng cường ảnh mạnh (và, trong các biến thể sau, thêm các hàm mất mát phụ trợ) khi huấn luyện học sinh trên dữ liệu gán nhãn giả, để học sinh không thể đơn giản ghi nhớ đầu ra chính xác của giáo viên — nó phải học thứ gì đó vững hơn để tái tạo chúng dưới sự biến dạng, đó mới là điều thực sự vắt ra được khả năng tổng quát hóa mới từ quá trình này, thay vì chỉ sao chép giáo viên.

Không thành phần nào trong số này khắc phục được vấn đề cơ bản của quy trình đồ chơi này (không có thông tin thực sự mới nào về các độ sâu trong [0.6, 0.9] ở bất cứ đâu trong tập dữ liệu này), nhưng ở quy mô thực, "thông tin thực sự mới" hiếm khi hoàn toàn vắng mặt trong một kho không nhãn đủ lớn và đa dạng — đó chính xác là điều kiện mà tự huấn luyện cần để đáng thực hiện.

### Bài tập

1. Đổi `depth_range` của kho không nhãn để khớp chính xác với `(0.4, 0.6)` của tập có nhãn, thay vì toàn bộ khoảng `(0.2, 0.9)`. Tự huấn luyện *giờ* có giúp ích không — và điều đó có ủng hộ khẳng định rằng giá trị của tự huấn luyện phụ thuộc hoàn toàn vào việc các nhãn thực (không quan sát được) của kho không nhãn có thực sự khác với những gì giáo viên đã tin hay không?
2. Thêm nhiễu điểm ảnh Gauss vào các ảnh không nhãn *trước khi* tạo nhãn giả nhưng huấn luyện học sinh trên các ảnh *sạch* với các nhãn suy ra từ ảnh nhiễu đó (hoặc ngược lại) — một sự thay thế thô sơ cho mẹo nhiễu loạn của Depth Anything. Việc ép sự không khớp này giữa thứ được gán nhãn giả và thứ được huấn luyện có làm thay đổi MAE kiểm tra của học sinh không?
3. Huấn luyện lại giáo viên trên một tập có nhãn bao phủ toàn bộ khoảng độ sâu `(0.2, 0.9)` (cùng 20 mẫu, chỉ lấy mẫu lại với `depth_range=(0.2, 0.9)`) thay vì khoảng hẹp. Tự huấn luyện có giúp giáo viên *này* cải thiện thêm trên cùng tập kiểm tra không, khi giờ ngay từ đầu không có khoảng cách thông tin cơ bản nào?